# 基于 Vision Transformer 的 CIFAR-10 图像分类实验

这是一个单文件 notebook 版本。提交时只需要提交本 notebook 和实验报告，不依赖项目中的其他 `.py` 文件。

Notebook 内置：CIFAR-10 手动下载/校验/解压、数据加载、CNN baseline、ViT、训练、评估、混淆矩阵、预测样例和结果汇总。

## 1. 导入依赖

In [ ]:
import csv
import hashlib
import json
import math
import os
import random
import tarfile
import tempfile
import time
import urllib.request
from datetime import datetime
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "vit_cifar10_mpl_cache"))
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, Markdown, display
from sklearn.metrics import classification_report, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("torchvision dataset module loaded")

## 2. 实验配置

服务器网络不好时，保持 `download=False`，先手动上传 `data/cifar-10-python.tar.gz`，再运行下一节的数据准备单元格。网络可用时可改为 `download=True`。

In [ ]:
RUN_CONFIG = {
    "seed": 42,
    "device": "auto",
    "data_root": Path("./data"),
    "output_root": Path("./outputs"),
    "image_size": 224,
    "val_fraction": 0.1,
    "num_workers": 2,
    "download": False,
    "download_timeout_seconds": 60,
    "train_subset": None,
    "val_subset": None,
    "test_subset": None,
}

MODEL_CONFIGS = {
    "cnn": {"epochs": 5, "batch_size": 16, "lr": 1e-3, "weight_decay": 1e-4},
    "vit": {
        "epochs": 3,
        "batch_size": 8,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "pretrained": True,
        "train_mode": "head_only",  # "head_only" or "full"
    },
}

RUN_CONFIG, MODEL_CONFIGS

## 3. CIFAR-10 手动下载、校验与解压

手动下载地址：`https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz`

把文件上传为 `data/cifar-10-python.tar.gz`，运行本节代码即可校验 MD5 并解压。

In [ ]:
CIFAR10_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
CIFAR10_ARCHIVE = "cifar-10-python.tar.gz"
CIFAR10_ARCHIVE_MD5 = "c58f30108f718f92721af3b95e74349a"
CIFAR10_FOLDER = "cifar-10-batches-py"
CIFAR10_REQUIRED_FILES = (
    "data_batch_1", "data_batch_2", "data_batch_3", "data_batch_4", "data_batch_5", "test_batch", "batches.meta"
)


def has_cifar10_files(data_root):
    data_dir = Path(data_root) / CIFAR10_FOLDER
    return all((data_dir / name).exists() for name in CIFAR10_REQUIRED_FILES)


def file_md5(path, chunk_size=1024 * 1024):
    digest = hashlib.md5()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def format_size(num_bytes):
    size = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024.0:
            return f"{size:.1f} {unit}"
        size /= 1024.0
    return f"{size:.1f} TB"


def download_cifar10_archive(data_root, timeout_seconds=60):
    data_root = Path(data_root)
    data_root.mkdir(parents=True, exist_ok=True)
    archive_path = data_root / CIFAR10_ARCHIVE
    partial_path = archive_path.with_suffix(archive_path.suffix + ".part")

    if archive_path.exists() and file_md5(archive_path) == CIFAR10_ARCHIVE_MD5:
        print(f"Found CIFAR-10 archive: {archive_path}")
        return archive_path
    if archive_path.exists():
        print(f"Existing archive has wrong MD5 and will be replaced: {archive_path}")
        archive_path.unlink()
    if partial_path.exists():
        partial_path.unlink()

    print(f"Downloading CIFAR-10 from {CIFAR10_URL}")
    request = urllib.request.Request(CIFAR10_URL, headers={"User-Agent": "Mozilla/5.0"})
    start = time.monotonic()
    last_report = start
    downloaded = 0
    total_bytes = None

    try:
        with urllib.request.urlopen(request, timeout=timeout_seconds) as response:
            content_length = response.headers.get("Content-Length")
            if content_length and content_length.isdigit():
                total_bytes = int(content_length)
            with partial_path.open("wb") as file:
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    file.write(chunk)
                    downloaded += len(chunk)
                    now = time.monotonic()
                    if now - last_report >= 2.0 or (total_bytes is not None and downloaded >= total_bytes):
                        elapsed = max(now - start, 1e-6)
                        speed = downloaded / elapsed
                        if total_bytes is None:
                            print(f"Downloading CIFAR-10: {format_size(downloaded)} at {format_size(int(speed))}/s")
                        else:
                            percent = downloaded / total_bytes * 100
                            print(
                                f"Downloading CIFAR-10: {format_size(downloaded)} / "
                                f"{format_size(total_bytes)} ({percent:.1f}%) at {format_size(int(speed))}/s"
                            )
                        last_report = now
    except Exception:
        if partial_path.exists():
            partial_path.unlink()
        raise

    partial_path.replace(archive_path)
    if file_md5(archive_path) != CIFAR10_ARCHIVE_MD5:
        archive_path.unlink()
        raise RuntimeError("Downloaded CIFAR-10 archive failed MD5 verification.")
    print(f"CIFAR-10 archive downloaded: {archive_path}")
    return archive_path


def extract_cifar10_archive(archive_path, data_root):
    archive_path = Path(archive_path)
    data_root = Path(data_root)
    data_root.mkdir(parents=True, exist_ok=True)
    data_root_resolved = data_root.resolve()
    print(f"Extracting CIFAR-10 archive to {data_root}")
    with tarfile.open(archive_path, "r:gz") as archive:
        for member in archive.getmembers():
            target_path = (data_root / member.name).resolve()
            try:
                target_path.relative_to(data_root_resolved)
            except ValueError as exc:
                raise RuntimeError(f"Unsafe path in archive: {member.name}") from exc
        archive.extractall(data_root)
    if not has_cifar10_files(data_root):
        raise RuntimeError("Archive extracted, but CIFAR-10 files were not found.")
    print(f"CIFAR-10 is ready: {data_root / CIFAR10_FOLDER}")


def ensure_cifar10_data(data_root, download=False, timeout_seconds=60):
    data_root = Path(data_root)
    archive_path = data_root / CIFAR10_ARCHIVE
    if has_cifar10_files(data_root):
        print(f"CIFAR-10 found locally: {data_root / CIFAR10_FOLDER}")
        return
    if archive_path.exists():
        actual_md5 = file_md5(archive_path)
        print(f"Found archive: {archive_path}")
        print(f"Actual MD5: {actual_md5}")
        if actual_md5 != CIFAR10_ARCHIVE_MD5:
            raise RuntimeError(f"MD5 mismatch. Expected {CIFAR10_ARCHIVE_MD5}, got {actual_md5}.")
        extract_cifar10_archive(archive_path, data_root)
        return
    if download:
        archive_path = download_cifar10_archive(data_root, timeout_seconds=timeout_seconds)
        extract_cifar10_archive(archive_path, data_root)
        return
    display(Markdown(
        "未发现 CIFAR-10 数据。请先手动下载并上传：\n\n"
        f"- 下载地址：`{CIFAR10_URL}`\n"
        f"- 上传位置：`{archive_path}`\n"
        f"- 期望 MD5：`{CIFAR10_ARCHIVE_MD5}`\n\n"
        "上传后重新运行本单元格。"
    ))


ensure_cifar10_data(
    RUN_CONFIG["data_root"],
    download=RUN_CONFIG["download"],
    timeout_seconds=RUN_CONFIG["download_timeout_seconds"],
)

## 4. 数据加载与通用工具

In [ ]:
CIFAR10_CLASSES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


def resolve_device(device_name="auto"):
    if device_name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(device_name)


def save_json(data, output_path):
    def ready(value):
        if isinstance(value, dict):
            return {key: ready(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [ready(item) for item in value]
        if isinstance(value, Path):
            return str(value)
        return value
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(ready(data), file, indent=2, ensure_ascii=False)


def build_transforms(image_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(image_size, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
    ])
    eval_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
    ])
    return train_transform, eval_transform


def limit_indices(indices, limit):
    if limit is None:
        return indices
    return indices[: max(0, min(limit, len(indices)))]


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed + worker_id)


def build_cifar10_dataloaders(data_root, batch_size, num_workers, image_size, val_fraction, seed, download, pin_memory, train_subset=None, val_subset=None, test_subset=None, download_timeout=60):
    if not 0 < val_fraction < 1:
        raise ValueError("val_fraction must be between 0 and 1")
    data_root = Path(data_root)
    ensure_cifar10_data(data_root, download=download, timeout_seconds=download_timeout)
    train_transform, eval_transform = build_transforms(image_size=image_size)
    train_dataset = datasets.CIFAR10(root=data_root, train=True, transform=train_transform, download=False)
    val_dataset = datasets.CIFAR10(root=data_root, train=True, transform=eval_transform, download=False)
    test_dataset = datasets.CIFAR10(root=data_root, train=False, transform=eval_transform, download=False)

    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(train_dataset), generator=generator).tolist()
    val_size = int(len(indices) * val_fraction)
    val_indices = limit_indices(indices[:val_size], val_subset)
    train_indices = limit_indices(indices[val_size:], train_subset)
    test_indices = limit_indices(list(range(len(test_dataset))), test_subset)

    common_kwargs = {
        "num_workers": num_workers,
        "pin_memory": pin_memory,
        "worker_init_fn": seed_worker,
        "persistent_workers": num_workers > 0,
    }
    loader_generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(Subset(train_dataset, train_indices), batch_size=batch_size, shuffle=True, generator=loader_generator, **common_kwargs)
    val_loader = DataLoader(Subset(val_dataset, val_indices), batch_size=batch_size, shuffle=False, **common_kwargs)
    test_loader = DataLoader(Subset(test_dataset, test_indices), batch_size=batch_size, shuffle=False, **common_kwargs)
    return train_loader, val_loader, test_loader, CIFAR10_CLASSES, len(train_indices), len(val_indices), len(test_indices)


def count_parameters(model, trainable_only=False):
    parameters = model.parameters()
    if trainable_only:
        parameters = (parameter for parameter in parameters if parameter.requires_grad)
    return sum(parameter.numel() for parameter in parameters)

## 5. 模型定义

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128, 128), nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


class PatchEmbedding(nn.Module):
    def __init__(self, image_size=224, patch_size=16, embed_dim=192):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.projection = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, images):
        tokens = self.projection(images)
        return tokens.flatten(2).permute(2, 0, 1)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=192, num_heads=3, mlp_dim=384, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(nn.Linear(embed_dim, mlp_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(mlp_dim, embed_dim), nn.Dropout(dropout))

    def forward(self, tokens):
        normalized = self.norm1(tokens)
        attention_output, _ = self.attention(normalized, normalized, normalized, need_weights=False)
        tokens = tokens + self.dropout(attention_output)
        return tokens + self.mlp(self.norm2(tokens))


class LightweightVisionTransformer(nn.Module):
    def __init__(self, num_classes=10, image_size=224, patch_size=16, embed_dim=192, depth=4, num_heads=3, mlp_dim=384, dropout=0.1):
        super().__init__()
        self.patch_embedding = PatchEmbedding(image_size=image_size, patch_size=patch_size, embed_dim=embed_dim)
        self.class_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.position_embedding = nn.Parameter(torch.zeros(self.patch_embedding.num_patches + 1, 1, embed_dim))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim=embed_dim, num_heads=num_heads, mlp_dim=mlp_dim, dropout=dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        self.init_weights()

    def init_weights(self):
        nn.init.normal_(self.class_token, std=0.02)
        nn.init.normal_(self.position_embedding, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, images):
        tokens = self.patch_embedding(images)
        class_tokens = self.class_token.expand(-1, tokens.size(1), -1)
        tokens = torch.cat((class_tokens, tokens), dim=0)
        tokens = self.dropout(tokens + self.position_embedding)
        for block in self.blocks:
            tokens = block(tokens)
        return self.head(self.norm(tokens)[0])


def set_train_mode(model, train_mode, head):
    if train_mode == "head_only":
        for parameter in model.parameters():
            parameter.requires_grad = False
        for parameter in head.parameters():
            parameter.requires_grad = True
    elif train_mode != "full":
        raise ValueError("train_mode must be 'full' or 'head_only'")


def build_vit_model(num_classes=10, pretrained=True, train_mode="full"):
    try:
        from torchvision.models import vit_b_16
        try:
            from torchvision.models import ViT_B_16_Weights
        except ImportError:
            ViT_B_16_Weights = None
        if ViT_B_16_Weights is not None:
            weights = ViT_B_16_Weights.DEFAULT if pretrained else None
            model = vit_b_16(weights=weights)
        else:
            model = vit_b_16(pretrained=pretrained)
        if hasattr(model, "heads") and hasattr(model.heads, "head"):
            in_features = model.heads.head.in_features
            model.heads.head = nn.Linear(in_features, num_classes)
            set_train_mode(model, train_mode, model.heads)
            return model
    except Exception as exc:
        print(f"Official torchvision ViT is unavailable, using lightweight ViT fallback. Reason: {exc}")

    effective_train_mode = "full" if train_mode == "head_only" else train_mode
    model = LightweightVisionTransformer(num_classes=num_classes)
    set_train_mode(model, effective_train_mode, model.head)
    return model

## 6. 训练、评估和绘图函数

In [ ]:
def create_experiment_dirs(output_root):
    experiment_name = datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_dir = Path(output_root) / experiment_name
    paths = {
        "experiment": experiment_dir,
        "figures": experiment_dir / "figures",
        "checkpoints": experiment_dir / "checkpoints",
        "logs": experiment_dir / "logs",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def run_epoch(model, loader, criterion, device, model_name, epoch, training, optimizer=None):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    phase = "train" if training else "val"
    progress = tqdm(loader, desc=f"{model_name} {phase} epoch {epoch}", leave=False)
    for images, targets in progress:
        images = images.to(device, non_blocking=device.type == "cuda")
        targets = targets.to(device, non_blocking=device.type == "cuda")
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                optimizer.step()
        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == targets).sum().item()
        total += batch_size
        progress.set_postfix(loss=total_loss / max(total, 1), acc=correct / max(total, 1))
    return {"loss": total_loss / max(total, 1), "accuracy": correct / max(total, 1)}


def write_history_csv(history, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "lr", "epoch_seconds"])
        writer.writeheader()
        writer.writerows(history)


def train_model(model, train_loader, val_loader, device, model_name, epochs, optimizer, scheduler, checkpoint_dir, log_dir):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    checkpoint_dir = Path(checkpoint_dir)
    log_dir = Path(log_dir)
    best_checkpoint = checkpoint_dir / f"{model_name}_best.pth"
    last_checkpoint = checkpoint_dir / f"{model_name}_last.pth"
    history_csv = log_dir / f"{model_name}_history.csv"
    best_val_accuracy = -1.0
    history = []
    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        train_stats = run_epoch(model, train_loader, criterion, device, model_name, epoch, True, optimizer)
        val_stats = run_epoch(model, val_loader, criterion, device, model_name, epoch, False)
        if scheduler is not None:
            scheduler.step()
        row = {
            "epoch": epoch,
            "train_loss": float(train_stats["loss"]),
            "train_acc": float(train_stats["accuracy"]),
            "val_loss": float(val_stats["loss"]),
            "val_acc": float(val_stats["accuracy"]),
            "lr": float(optimizer.param_groups[0]["lr"]),
            "epoch_seconds": float(time.perf_counter() - start),
        }
        history.append(row)
        write_history_csv(history, history_csv)
        is_best = row["val_acc"] > best_val_accuracy
        if is_best:
            best_val_accuracy = row["val_acc"]
        checkpoint = {"epoch": epoch, "model_state_dict": model.state_dict(), "best_val_accuracy": best_val_accuracy, "history": history}
        torch.save(checkpoint, last_checkpoint)
        if is_best:
            torch.save(checkpoint, best_checkpoint)
        print(f"{model_name} epoch {epoch:02d}/{epochs} | train_acc={row['train_acc']:.4f} val_acc={row['val_acc']:.4f}")
    return history, best_checkpoint, best_val_accuracy


def unnormalize(image):
    array = image.detach().cpu().numpy()
    mean = np.asarray(CIFAR10_MEAN, dtype=np.float32).reshape(3, 1, 1)
    std = np.asarray(CIFAR10_STD, dtype=np.float32).reshape(3, 1, 1)
    array = np.clip(array * std + mean, 0.0, 1.0)
    return np.transpose(array, (1, 2, 0))


def plot_loss_curve(history, output_path, title):
    epochs = [row["epoch"] for row in history]
    plt.figure(figsize=(7, 4.5), dpi=150)
    plt.plot(epochs, [row["train_loss"] for row in history], marker="o", label="Train loss")
    plt.plot(epochs, [row["val_loss"] for row in history], marker="s", label="Validation loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(title); plt.grid(alpha=0.25); plt.legend(); plt.tight_layout(); plt.savefig(output_path); plt.close()


def plot_accuracy_curve(history, output_path, title):
    epochs = [row["epoch"] for row in history]
    plt.figure(figsize=(7, 4.5), dpi=150)
    plt.plot(epochs, [row["train_acc"] for row in history], marker="o", label="Train accuracy")
    plt.plot(epochs, [row["val_acc"] for row in history], marker="s", label="Validation accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.ylim(0, 1); plt.title(title); plt.grid(alpha=0.25); plt.legend(); plt.tight_layout(); plt.savefig(output_path); plt.close()


def plot_confusion_matrix_image(matrix, class_names, output_path, title):
    matrix = np.asarray(matrix)
    plt.figure(figsize=(8, 7), dpi=150)
    image = plt.imshow(matrix, interpolation="nearest", cmap="Blues")
    plt.colorbar(image, fraction=0.046, pad=0.04)
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right"); plt.yticks(ticks, class_names)
    plt.xlabel("Predicted label"); plt.ylabel("True label"); plt.title(title)
    threshold = matrix.max() / 2 if matrix.size and matrix.max() > 0 else 0
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = int(matrix[row, col])
            plt.text(col, row, str(value), ha="center", va="center", color="white" if value > threshold else "black", fontsize=7)
    plt.tight_layout(); plt.savefig(output_path); plt.close()


def plot_prediction_samples(samples, class_names, output_path, title):
    if not samples:
        return
    cols = min(4, len(samples))
    rows = math.ceil(len(samples) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.0, rows * 3.2), dpi=150)
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for axis in axes.ravel():
        axis.axis("off")
    for axis, sample in zip(axes.ravel(), samples):
        color = "#137333" if sample["target"] == sample["prediction"] else "#b3261e"
        axis.imshow(unnormalize(sample["image"]))
        axis.set_title(f"T: {class_names[sample['target']]}\nP: {class_names[sample['prediction']]} ({sample['confidence']:.2f})", color=color, fontsize=9)
        axis.axis("off")
    fig.suptitle(title, fontsize=12); plt.tight_layout(); plt.savefig(output_path); plt.close(fig)


def evaluate_model(model, data_loader, device, class_names, model_name, checkpoint_path, figures_dir, logs_dir, num_samples=12):
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device); model.eval()
    criterion = nn.CrossEntropyLoss()
    all_targets, all_predictions, samples = [], [], []
    total_loss, correct, total, seen = 0.0, 0, 0, 0
    rng = random.Random(RUN_CONFIG["seed"])
    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc=f"{model_name} test", leave=False):
            images = images.to(device, non_blocking=device.type == "cuda")
            targets = targets.to(device, non_blocking=device.type == "cuda")
            logits = model(images)
            loss = criterion(logits, targets)
            probs = torch.softmax(logits, dim=1)
            confidences, predictions = probs.max(dim=1)
            total_loss += loss.item() * targets.size(0)
            correct += (predictions == targets).sum().item()
            total += targets.size(0)
            all_targets.extend(targets.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())
            for index in range(targets.size(0)):
                seen += 1
                candidate = {"image": images[index].cpu(), "target": int(targets[index].cpu()), "prediction": int(predictions[index].cpu()), "confidence": float(confidences[index].cpu())}
                if len(samples) < num_samples:
                    samples.append(candidate)
                else:
                    replacement = rng.randint(0, seen - 1)
                    if replacement < num_samples:
                        samples[replacement] = candidate
    labels = list(range(len(class_names)))
    report_dict = classification_report(all_targets, all_predictions, labels=labels, target_names=list(class_names), output_dict=True, zero_division=0)
    report_text = classification_report(all_targets, all_predictions, labels=labels, target_names=list(class_names), zero_division=0)
    matrix = confusion_matrix(all_targets, all_predictions, labels=labels)
    accuracy = correct / max(total, 1)
    metrics = {"accuracy": accuracy, "test_loss": total_loss / max(total, 1), "macro_precision": report_dict["macro avg"]["precision"], "macro_recall": report_dict["macro avg"]["recall"], "macro_f1": report_dict["macro avg"]["f1-score"], "classification_report": report_dict, "confusion_matrix": matrix.tolist()}
    save_json(metrics, logs_dir / f"{model_name}_metrics.json")
    (logs_dir / f"{model_name}_classification_report.txt").write_text(report_text, encoding="utf-8")
    plot_confusion_matrix_image(matrix, class_names, figures_dir / f"{model_name}_confusion_matrix.png", f"{model_name.upper()} confusion matrix")
    plot_prediction_samples(samples, class_names, figures_dir / f"{model_name}_prediction_samples.png", f"{model_name.upper()} prediction samples")
    print(f"{model_name} test accuracy: {accuracy:.4f}")
    print(report_text)
    return metrics

## 7. 初始化实验目录

In [ ]:
seed_everything(RUN_CONFIG["seed"])
device = resolve_device(RUN_CONFIG["device"])
paths = create_experiment_dirs(RUN_CONFIG["output_root"])
save_json({"run_config": RUN_CONFIG, "model_configs": MODEL_CONFIGS, "device": str(device), "experiment_dir": str(paths["experiment"])}, paths["logs"] / "notebook_run_config.json")
print(f"Device: {device}")
print(f"Experiment directory: {paths['experiment']}")

## 8. 单模型训练与评估

In [ ]:
def build_model(model_name):
    if model_name == "cnn":
        return SimpleCNN(num_classes=10)
    if model_name == "vit":
        config = MODEL_CONFIGS["vit"]
        return build_vit_model(num_classes=10, pretrained=config["pretrained"], train_mode=config["train_mode"])
    raise ValueError(f"Unsupported model: {model_name}")


def run_one_model(model_name):
    config = MODEL_CONFIGS[model_name]
    train_loader, val_loader, test_loader, class_names, train_size, val_size, test_size = build_cifar10_dataloaders(
        data_root=RUN_CONFIG["data_root"], batch_size=config["batch_size"], num_workers=RUN_CONFIG["num_workers"], image_size=RUN_CONFIG["image_size"],
        val_fraction=RUN_CONFIG["val_fraction"], seed=RUN_CONFIG["seed"], download=RUN_CONFIG["download"], pin_memory=device.type == "cuda",
        train_subset=RUN_CONFIG["train_subset"], val_subset=RUN_CONFIG["val_subset"], test_subset=RUN_CONFIG["test_subset"], download_timeout=RUN_CONFIG["download_timeout_seconds"])
    print(f"{model_name} data | train={train_size} val={val_size} test={test_size}")
    model = build_model(model_name)
    total_params = count_parameters(model)
    trainable_params = count_parameters(model, trainable_only=True)
    print(f"{model_name} parameters: total={total_params:,}, trainable={trainable_params:,}")
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, config["epochs"]))
    history, best_checkpoint, best_val_accuracy = train_model(model, train_loader, val_loader, device, model_name, config["epochs"], optimizer, scheduler, paths["checkpoints"], paths["logs"])
    plot_loss_curve(history, paths["figures"] / f"{model_name}_loss_curve.png", f"{model_name.upper()} loss curve")
    plot_accuracy_curve(history, paths["figures"] / f"{model_name}_acc_curve.png", f"{model_name.upper()} accuracy curve")
    metrics = evaluate_model(model, test_loader, device, class_names, model_name, best_checkpoint, paths["figures"], paths["logs"])
    return {"accuracy": metrics["accuracy"], "macro_precision": metrics["macro_precision"], "macro_recall": metrics["macro_recall"], "macro_f1": metrics["macro_f1"], "parameters": total_params, "trainable_parameters": trainable_params, "best_val_accuracy": best_val_accuracy}

## 9. 运行 CNN 和 ViT

In [ ]:
results = {}
results["cnn"] = run_one_model("cnn")
results["vit"] = run_one_model("vit")
results

## 10. 汇总和展示结果

In [ ]:
def write_comparison_csv(results, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fields = ["model", "accuracy", "macro_precision", "macro_recall", "macro_f1", "parameters", "trainable_parameters", "best_val_accuracy"]
    with output_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        for model_name, result in results.items():
            writer.writerow({"model": model_name, **result})


def plot_model_comparison(results, output_path):
    names = list(results.keys())
    x = np.arange(len(names))
    width = 0.36
    plt.figure(figsize=(7, 4.5), dpi=150)
    plt.bar(x - width / 2, [results[name]["accuracy"] for name in names], width, label="Accuracy")
    plt.bar(x + width / 2, [results[name]["macro_f1"] for name in names], width, label="Macro F1")
    plt.xticks(x, [name.upper() for name in names]); plt.ylim(0, 1); plt.ylabel("Score"); plt.title("CNN vs ViT on CIFAR-10"); plt.grid(axis="y", alpha=0.25); plt.legend(); plt.tight_layout(); plt.savefig(output_path); plt.close()


save_json(results, paths["logs"] / "notebook_comparison_summary.json")
write_comparison_csv(results, paths["logs"] / "notebook_comparison_summary.csv")
plot_model_comparison(results, paths["figures"] / "notebook_model_comparison.png")

for model_name, result in results.items():
    print(f"{model_name.upper()} | acc={result['accuracy']:.4f} macro_f1={result['macro_f1']:.4f} params={result['parameters']:,}")
print(f"Outputs saved to: {paths['experiment']}")

In [ ]:
figure_names = [
    "cnn_loss_curve.png", "cnn_acc_curve.png", "cnn_confusion_matrix.png", "cnn_prediction_samples.png",
    "vit_loss_curve.png", "vit_acc_curve.png", "vit_confusion_matrix.png", "vit_prediction_samples.png",
    "notebook_model_comparison.png",
]

for figure_name in figure_names:
    figure_path = paths["figures"] / figure_name
    if figure_path.exists():
        display(Markdown(f"### {figure_name}"))
        display(Image(filename=str(figure_path)))